# extract_radiomics example

## 1. Setup

In [1]:
import os
import sys

# 프로젝트 root 기준으로 path 추가
sys.path.append("../src")

from h5radiomics.extract_radiomics import (
    build_radiomics_extractor,
    extract_radiomics,
)

import pandas as pd

## 2. Input 설정

In [2]:
sample_id = "TENX95"

h5_path = f"../h5/{sample_id}.h5"
output_root = f"../outputs/notebook_test/{sample_id}"

os.makedirs(output_root, exist_ok=True)

print("H5 path:", h5_path)
print("Output:", output_root)

H5 path: ../h5/TENX95.h5
Output: ../outputs/notebook_test/TENX95


## 3. Extractor 생성

In [3]:
extractor = build_radiomics_extractor(
    classes=["firstorder", "glcm", "glrlm", "glszm", "gldm", "ngtdm"],
    filters=["Original"],
    label=255,
)

## 4. Radiomics 실행

In [4]:
result = extract_radiomics(
    h5_path=h5_path,
    output_root=output_root,
    extractor=extractor,
    save_patches=False,
    num_workers=0,  # notebook에서는 0 추천
)

[Processing patches]: 100%|██████████| 7760/7760 [12:56<00:00,  9.99it/s]


## 5. 결과 DataFrame

In [5]:
df = pd.DataFrame(result["rows"])

print("Total patches:", result["total_num_patches"])
print("Shape:", df.shape)

df.head()

Total patches: 7760
Shape: (7760, 123)


,patch_idx,barcode,color_path,gray_path,mask_path,x,y,status,diagnostics_Versions_PyRadiomics,diagnostics_Versions_Numpy,...,original_gldm_LargeDependenceLowGrayLevelEmphasis,original_gldm_LowGrayLevelEmphasis,original_gldm_SmallDependenceEmphasis,original_gldm_SmallDependenceHighGrayLevelEmphasis,original_gldm_SmallDependenceLowGrayLevelEmphasis,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
0,0,005x028,,,,2616,13425,ok,v3.0.1,1.26.4,...,1.3669642857142856,0.3705357142857143,0.5041666666666667,1.8327380952380956,0.17202380952380955,8.358823529411767,0.039408866995073885,0.18125000000000002,0.022159712099125362,0.043478260869565216
1,1,005x029,,,,2616,13896,ok,v3.0.1,1.26.4,...,0.293056258790436,0.0680464135021097,0.4676511954992968,14.040365682137834,0.030922722300359435,0.6336177080931519,0.03593522561863173,4.0143251015744,0.024054567691399666,1.0583442455987198
2,2,005x030,,,,2616,14367,ok,v3.0.1,1.26.4,...,0.30043432883750804,0.07142341040462427,0.4725433526011561,14.163615928066795,0.033959448369371294,0.41299588695767536,0.0555778652317083,2.9741087953326586,0.02392278537046347,1.425230552001059
3,3,005x031,,,,2616,14837,ok,v3.0.1,1.26.4,...,0.4423285590277778,0.07356770833333334,0.2485894097222222,3.6061197916666665,0.022489571277006175,0.9756163397516646,0.04721940422392327,0.7130242950299186,0.00249350443482399,0.5819444444444445
4,4,006x018,,,,3087,8720,ok,v3.0.1,1.26.4,...,0.2837041884816754,0.07100785340314136,0.40997673065735896,6.0823152995927865,0.03152874733372116,0.9672852724244588,0.04698069118189645,1.001461501488446,0.0026571513020261137,0.5859220477021525


## 6. Feature만 확인

In [6]:
meta_cols = [
    "patch_idx", "barcode", "color_path", "gray_path",
    "mask_path", "x", "y", "status"
]

feature_cols = [c for c in df.columns if c not in meta_cols]

print("Num features:", len(feature_cols))
feature_cols[:10]

Num features: 115


['diagnostics_Versions_PyRadiomics',
 'diagnostics_Versions_Numpy',
 'diagnostics_Versions_SimpleITK',
 'diagnostics_Versions_PyWavelet',
 'diagnostics_Versions_Python',
 'diagnostics_Configuration_Settings',
 'diagnostics_Configuration_EnabledImageTypes',
 'diagnostics_Image-original_Hash',
 'diagnostics_Image-original_Dimensionality',
 'diagnostics_Image-original_Spacing']

## 7. 간단한 통계 확인

In [7]:
df[feature_cols].describe().T.head()

,count,mean,std,min,25%,50%,75%,max
diagnostics_Image-original_Mean,7756.0,199.995043,9.740959,158.911392,194.672259,198.746941,203.241699,238.695671
diagnostics_Image-original_Minimum,7756.0,18.300670,19.955950,0.000000,11.000000,13.000000,18.000000,209.000000
diagnostics_Image-original_Maximum,7756.0,252.335095,2.066717,235.000000,251.000000,253.000000,254.000000,255.000000
diagnostics_Mask-original_VoxelNum,7756.0,38738.749226,9422.481939,61.000000,37131.500000,41581.000000,44413.500000,49792.000000
diagnostics_Mask-original_VolumeNum,7756.0,36.304281,46.542704,1.000000,15.000000,26.000000,45.000000,1013.000000


## 8. 특정 patch 확인

In [8]:
df[df["status"] == "ok"].iloc[0]

patch_idx                                       0
barcode                                   005x028
color_path                                       
gray_path                                        
mask_path                                        
                                     ...         
original_ngtdm_Busyness         8.358823529411767
original_ngtdm_Coarseness    0.039408866995073885
original_ngtdm_Complexity     0.18125000000000002
original_ngtdm_Contrast      0.022159712099125362
original_ngtdm_Strength      0.043478260869565216
Name: 0, Length: 123, dtype: object